# DS2002 · Walmart Warm-Up SKU Mess

**Lab — 2026-10-16 · Fall 2026**  

---

## Lab 08 — Walmart Warm-Up (SKU Mess)

This lab uses the **real midterm data** in `../06-midterm-walmart/data/`. It's a warm-up so you hit the ground running on the project. On Kaggle/Colab, upload the data folder or adjust the path.

*If the file isn't found, the cell falls back to a small inline sample so the notebook still runs.*

In [1]:
import pandas as pd
PATH = 'product_catalog.csv'
try:
    cat = pd.read_csv(PATH)
except FileNotFoundError:
    from io import StringIO
    cat = pd.read_csv(StringIO('sku,product_name,category\nPT-12,pop-tarts strawberry,Breakfast Foods\nSKU#459812,Strawberry Pop-Tart,Breakfast\nPOPTART-STRAW,Pop-Tarts Strawberry,Breakfast'))
cat.head()

,sku,product_name,category,subcategory,unit_cost,supplier,package_size
0,PT-12,pop-tarts strawberry,Breakfast Foods,NaN,1.93,Johnson & Johnson,4-pack
1,SKU#459812,Strawberry Pop-Tart,Breakfast,Pop-Tarts,2.26,Roku Inc,4pk
2,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast,NaN,2.11,Sara Lee,6 pack
3,PT_STRAWBERRY,pop-tarts strawberry,breakfast,Pop-Tarts,NaN,Campbell's,4 pack
4,WTR-24,BOTTLED WATER 24PK,Emergency Supplies,Bottled,2.64,Sara Lee,1 pack


### Q1 — How many rows look like Pop-Tart products?

Filter on `product_name` containing `pop`, case-insensitively. Print the count and the matching rows.

In [2]:
pops = cat[cat.product_name.str.lower().str.contains('pop')]

print(len(pops))

print(pops)


4
             sku          product_name         category subcategory  \
0          PT-12  pop-tarts strawberry  Breakfast Foods         NaN   
1     SKU#459812   Strawberry Pop-Tart        Breakfast   Pop-Tarts   
2  POPTART-STRAW  Pop-Tarts Strawberry        Breakfast         NaN   
3  PT_STRAWBERRY  pop-tarts strawberry        breakfast   Pop-Tarts   

   unit_cost           supplier package_size  
0       1.93  Johnson & Johnson       4-pack  
1       2.26           Roku Inc          4pk  
2       2.11           Sara Lee       6 pack  
3        NaN         Campbell's       4 pack  


### Q2 — Which SKUs and names all mean *strawberry* Pop-Tart?

List them. Then note which Pop-Tart rows are **not** strawberry — those must stay separate, and saying so is part of the answer.

In [3]:
name = cat.product_name.str.lower().str.replace(r'[^a-z]', '', regex=True)
sku = cat.sku.str.lower().str.replace(r'[^a-z]', '', regex=True)
broad = cat[name.str.contains('poptart') | sku.str.contains('poptart')]

straw = broad.product_name.str.lower().str.contains('straw')

print("Strawberry Pop-Tart variants:")
print(broad[straw][['sku', 'product_name']])

print("\nPop-Tart rows that are NOT strawberry:")
print(broad[~straw][['sku', 'product_name']])

cat[cat.subcategory.str.contains('Pop-Tarts', na=False) | cat.product_name.str.lower().str.contains('tart|toast')]

Strawberry Pop-Tart variants:
             sku          product_name
0          PT-12  pop-tarts strawberry
1     SKU#459812   Strawberry Pop-Tart
2  POPTART-STRAW  Pop-Tarts Strawberry
3  PT_STRAWBERRY  pop-tarts strawberry

Pop-Tart rows that are NOT strawberry:
Empty DataFrame
Columns: [sku, product_name]
Index: []


,sku,product_name,category,subcategory,unit_cost,supplier,package_size
0,PT-12,pop-tarts strawberry,Breakfast Foods,NaN,1.93,Johnson & Johnson,4-pack
1,SKU#459812,Strawberry Pop-Tart,Breakfast,Pop-Tarts,2.26,Roku Inc,4pk
2,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast,NaN,2.11,Sara Lee,6 pack
3,PT_STRAWBERRY,pop-tarts strawberry,breakfast,Pop-Tarts,NaN,Campbell's,4 pack


**Pop-Tart rows that are a different product:** none by flavor. I searched product name for pop and product name and sku for poptart; searched subcategory for Pop-Tarts, and searched names for tart or toast, but received the same output as earlier. If a different flavor existed, it would need a unique SKU since merging into strawberry would misattribute differing flavor sales. One row, POPTART-STRAW, is strawberry but a 6 pack rather than a 4 pack like the others, so I kept it as its own SKU.

### Q3 — Build the mapping

A dict from each variant SKU to one canonical SKU. Then apply it in a way that leaves unmapped SKUs as themselves rather than turning them into `NaN` — that single mistake silently deletes most of the catalog.

In [4]:
canonical = 'SKU#459812'  # only 4-pack row with complete data

sku_map = {'PT-12': canonical, 'SKU#459812': canonical, 'PT_STRAWBERRY': canonical}  # POPTART-STRAW (6 pack) kept separate

cat['canonical_sku'] = cat['sku'].map(sku_map).fillna(cat['sku'])

print(cat['canonical_sku'].isna().sum(), "nulls")
print(cat[cat['sku'] != cat['canonical_sku']][['sku', 'package_size', 'canonical_sku']])

print("Flagged, not merged (same flavor, 6 pack):")
print(cat[cat['sku'] == 'POPTART-STRAW'][['sku', 'package_size']])

print(cat['canonical_sku'].value_counts().head())

0 nulls
             sku package_size canonical_sku
0          PT-12       4-pack    SKU#459812
3  PT_STRAWBERRY       4 pack    SKU#459812
Flagged, not merged (same flavor, 6 pack):
             sku package_size
2  POPTART-STRAW       6 pack
canonical_sku
SKU#459812       3
POPTART-STRAW    1
WTR-24           1
WATER24PK        1
WTR_24PK         1
Name: count, dtype: int64


### Q4 — Normalize `category`

Strip and title-case, then show the distinct values before and after with counts. Note any pair that is still two labels for one thing — `Breakfast` and `Breakfast Foods` are not the same string but they are the same category.

In [5]:
print("BEFORE:")
print(cat['category'].value_counts(dropna=False))

cat['category_clean'] = cat['category'].str.strip().str.title()

print("AFTER:")
print(cat['category_clean'].value_counts(dropna=False))

BEFORE:
category
Emergency              5
Emergency Supplies     3
EMERGENCY              3
ENTERTAINMENT          3
GROCERY                3
Breakfast              2
entertainment          2
household              2
ELECTRONICS            2
health                 2
Baby                   2
Electronics            1
ALCOHOL                1
Alcoholic Beverages    1
breakfast              1
Breakfast Foods        1
Groceries              1
Grocery                1
electronics            1
snacks                 1
grocery                1
HOUSEHOLD              1
Snacks                 1
Household              1
baby                   1
Baby Products          1
emergency              1
Name: count, dtype: int64
AFTER:
category_clean
Emergency              9
Entertainment          5
Grocery                5
Electronics            4
Household              4
Emergency Supplies     3
Breakfast              3
Baby                   3
Snacks                 2
Health                 2
Breakfast 

Two Labels: Breakfast Foods and Breakfast, Emergency and Emergency Supplies, Alcohol and Alcoholic Beverages, Baby and Baby Products, Grocery and Groceries

### Q5 — Find the fragmentation you have not spotted yet

Hand-scanning a catalog does not scale. **TODO:** build a candidate list programmatically — normalize each product name (lowercase, strip punctuation, sort the words) and group by that, then show any group holding more than one SKU.

This is the workflow from Monday's lecture, and it is how you will actually approach the full catalog on the midterm.

In [6]:
def normalize(name):
    cleaned = name.lower().replace('-', ' ').replace('(', ' ').replace(')', ' ')
    return ' '.join(sorted(cleaned.split()))

cat['normalize'] = cat['product_name'].apply(normalize)

# Count distinct SKUs per normalized name; keep names shared by more than one SKU
groups = cat.groupby('normalize')['sku'].nunique()
candidates = cat[cat['normalize'].isin(groups[groups > 1].index)]

print(candidates.sort_values('normalize')[['normalize', 'sku', 'product_name']])

               normalize            sku           product_name
12    24 beer light pack        BEER-24   Light Beer (24-pack)
13    24 beer light pack       LTBEER24   Light Beer (24-pack)
8    4 aa batteries pack        BAT-AA4    AA Batteries 4-Pack
9    4 aa batteries pack    BATT_AA_4PK    AA Batteries 4-Pack
30         aid first kit   FIRSTAID-KIT          First Aid Kit
29         aid first kit        FAID-01          First-Aid Kit
15   board game monopoly    MONOPOLY-01    Monopoly Board Game
14   board game monopoly        BG-MONO    Monopoly Board Game
16   board game monopoly      GAME-MONO  Board Game - Monopoly
24           bread white     WHITEBREAD            WHITE BREAD
23           bread white       BREAD-WH            WHITE BREAD
25         butter peanut          PB-16          PEANUT BUTTER
26         butter peanut     PNUTBTR-16          Peanut Butter
28     candles emergency    CANDLE-EMRG      Emergency Candles
27     candles emergency        CNDL-EM    Candles (Eme

### Q6 — Prove the mapping is safe

**TODO:** assert that applying your mapping produced no nulls in the canonical column and did not change the number of rows. Consolidation renames; it never adds or removes.

In [7]:
original = pd.read_csv(PATH)

assert cat['canonical_sku'].isna().sum() == 0, "canonical_sku has nulls"
assert len(cat) == len(original), "row count changed"

print("Mapping is safe:", len(cat), "rows and 0 nulls in canonical_sku")

Mapping is safe: 45 rows and 0 nulls in canonical_sku


### Write-up

**a)** Name two product families beyond Pop-Tarts that appear fragmented, and how you would group them.

**b)** Where does consolidation stop being a data-cleaning task and become a business decision? Give one example from this catalog where you genuinely are not sure, and say who you would ask.

a) Two product families that still appear fragmented are the "aid first kit"s (FIRSTAID-KIT, FAID-01) and "bread white"s (BREAD-WH, WHITEBREAD). Both have two seemingly identical products with different SKUs. While the normalized column appears the same, I would double check the product_name column and that the package_size matches to confirm they align before standardizing the SKUs. Once confirmed, I would map each variant SKU to one canonical SKU, the same way I did for the Pop-Tarts.

b) Clarity of consolidation becomes a judgment call in this dataset when there are different sizes of the packs: 4-pack, 4pk, 6 pack, and 4 pack. Three of these (4-pack, 4pk, 4 pack) are the same size spelled differently, but the 6 pack is a genuinely different size. This could make a big difference if the store was trying to track costs, sales, and inventory, and equating a smaller size to a larger size. It must be determined whether a 6 pack is comparable to a 4 pack. I believe this decision should be delegated to a store manager, more specifically of the products, like dry-goods managers, to clarify the difference.